In [1]:
import gymnasium as gym
import numpy as np
from tensorflow.keras.models import load_model
from collections import deque
import cv2 as cv
import ale_py
import time

In [2]:
consecutive = 4
rows = 84
cols = 84

env = gym.make('ALE/Breakout-v5', render_mode='rgb_array')

actions = list(range(env.action_space.n))
n_actions = len(actions)

print("Action space:", env.action_space)
print("Actions:", actions)

try:
    print("Action meanings:", env.unwrapped.get_action_meanings())
except:
    print("Action meanings 확인 불가")

Action space: Discrete(4)
Actions: [0, 1, 2, 3]
Action meanings: ['NOOP', 'FIRE', 'RIGHT', 'LEFT']


In [3]:
def preprocess(img):

    crop = img[30:200, :, :]
    gray = cv.cvtColor(crop, cv.COLOR_RGB2GRAY)
    resized = cv.resize(gray, dsize=(rows, cols))

    return resized.reshape(rows, cols, 1)

In [4]:
model = load_model("fbreakout_checkpoint.keras")
# print("model loaded")

In [5]:
greedy_select = lambda x: np.random.choice(
    np.argwhere(x == np.max(x)).flatten()
)

def get_lives(env, info):
    if isinstance(info, dict) and "lives" in info:
        return info["lives"]

    try:
        return env.unwrapped.ale.lives()
    except:
        return 0


def play_match(env):
    obs, info = env.reset()

    score = 0
    step_count = 0
    action_count = {}

    action_meanings = env.unwrapped.get_action_meanings()
    print("Action meanings:", action_meanings)
    print("Actions:", actions)

    if "FIRE" in action_meanings:
        fire_action = action_meanings.index("FIRE")
    else:
        fire_action = 1

    obs, reward, terminated, truncated, info = env.step(fire_action)
    lives = get_lives(env, info)

    imem = deque(maxlen=consecutive)

    for _ in range(consecutive):
        imem.append(preprocess(obs))

    s = np.transpose(
        np.squeeze(np.asarray(imem)),
        (1, 2, 0)
    )

    while True:
        y_hat = model.predict(
            s.reshape(1, rows, cols, consecutive) / 255.0,
            verbose=0
        )[0]

        a = greedy_select(y_hat)

        action_count[a] = action_count.get(a, 0) + 1

        if step_count % 100 == 0:
            print(
                "step:",
                step_count,
                "Q:",
                np.round(y_hat, 3),
                "selected:",
                a,
                action_meanings[actions[a]] if actions[a] < len(action_meanings) else actions[a]
            )

        obs, reward, terminated, truncated, info = env.step(actions[a])
        score += reward
        step_count += 1

        new_lives = get_lives(env, info)

        if new_lives < lives:
            lives = new_lives

            if not (terminated or truncated):
                obs, reward, terminated, truncated, info = env.step(fire_action)
                score += reward

        imem.append(preprocess(obs))

        s = np.transpose(
            np.squeeze(np.asarray(imem)),
            (1, 2, 0)
        )

        frame = env.render()

        cv.imshow(
            "Breakout DQN Play",
            cv.cvtColor(frame, cv.COLOR_RGB2BGR)
        )

        key = cv.waitKey(1)

        if key == ord("q"):
            break

        if terminated or truncated:
            break

    print("score =", score)
    print("action_count =", action_count)
    cv.destroyAllWindows()

In [6]:
play_match(env)

env.close()

if cv.waitKey(0) == ord("q"):
    cv.destroyAllWindows()

Action meanings: ['NOOP', 'FIRE', 'RIGHT', 'LEFT']
Actions: [0, 1, 2, 3]
step: 0 Q: [2.325 2.336 2.353 2.334] selected: 2 RIGHT
step: 100 Q: [1.959 1.951 1.947 1.954] selected: 0 NOOP
step: 200 Q: [1.857 1.863 1.683 1.857] selected: 1 FIRE
step: 300 Q: [1.395 1.53  1.93  1.548] selected: 2 RIGHT
score = 12.0
action_count = {np.int64(2): 94, np.int64(3): 145, np.int64(0): 23, np.int64(1): 51}
